# Notebook 3 — Model Training (Session Chaining)
**RSNA Intracranial Hemorrhage Detection — EfficientNet-B0**

This notebook handles training across multiple Kaggle sessions.
Each session trains a fixed number of epochs, saves a checkpoint,
and the next session resumes from where the previous one left off.

### Session chaining workflow
| Session | `PREV_CHECKPOINT_DIR` | Trains epochs |
|---------|----------------------|---------------|
| 1 (first run) | `None` (starts fresh) | 0 → N-1 |
| 2 | `/kaggle/input/<nb3-session1-name>` | N → 2N-1 |
| 3 | `/kaggle/input/<nb3-session2-name>` | 2N → 3N-1 |

> **Before each session**: update `PREV_CHECKPOINT_DIR` to point to the
> previous session's notebook output name.

### Key improvements
- **Patient-level split**: Train/val split is done at the patient level (NB02).
  No slices from the same patient appear in both sets.
- **Dataset normalization**: Uses dataset-specific mean/std from NB02 (with ImageNet fallback).
- **NaN divergence guard**: Training aborts early if loss becomes NaN/Inf.
- **Backbone freezing**: Classifier head trains alone for the first N epochs,
  then backbone unfreezes with a lower learning rate (discriminative LR).
- **Early stopping**: Halts training when val_loss hasn't improved for `PATIENCE` epochs.
- **Discriminative LR**: Backbone uses `BASE_LR × BACKBONE_LR_FACTOR` to prevent

  catastrophic forgetting of ImageNet features.- Previous session checkpoint (all sessions except the first)

- Preprocessing cache (Notebook 02 output) — contains `cache/` NPY arrays, `manifest.csv`, `normalization_stats.json`
### Required input datasets

In [ ]:
from pathlib import Path
import os
import torch
import json

# ═══════════════════════════════════════════════════════════════════════════
# ██  CONFIG — edit these values at the start of each session  ██
# ═══════════════════════════════════════════════════════════════════════════

PREV_CHECKPOINT_DIR = None
N_EPOCHS_THIS_SESSION = 5
TOTAL_EPOCHS = 20

BATCH_SIZE   = 16
BASE_LR      = 1e-4
WEIGHT_DECAY = 1e-5
NUM_WORKERS  = 4
IMG_SIZE     = 256
SEED         = 42

ARCH = 'efficientnet_b0'  # or 'resnet50'

# ─── Anti-overfitting controls ───────────────────────────────────────────
FREEZE_BACKBONE_EPOCHS = 2    # Train only classifier head for first N epochs
BACKBONE_LR_FACTOR     = 0.1  # After unfreezing: backbone LR = BASE_LR × this
PATIENCE               = 5    # Early stopping: epochs without val_loss improvement
DROPOUT                = 0.4  # Classifier head dropout (default was 0.3)

# Path to NB02 output dataset
CACHE_INPUT_DIR = Path('/kaggle/input/notebooks/harshitghosh/nb02eda')  # <-- verify this name
MANIFEST_PATH   = CACHE_INPUT_DIR / 'manifest.csv'
NPY_CACHE_DIR   = CACHE_INPUT_DIR / 'cache'
STATS_PATH      = CACHE_INPUT_DIR / 'normalization_stats.json'

CHECKPOINT_PATH  = Path('/kaggle/working/checkpoint.pth')
METRICS_LOG_PATH = Path('/kaggle/working/training_metrics.csv')

# ═══════════════════════════════════════════════════════════════════════════
# ██  ENVIRONMENT INFO  ██
# ═══════════════════════════════════════════════════════════════════════════

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# ═══════════════════════════════════════════════════════════════════════════
# ██  VERIFY NB02 OUTPUT MOUNT  ██
# ═══════════════════════════════════════════════════════════════════════════

print("\nChecking mounted input datasets...")
print("Available input folders:")
print(os.listdir('/kaggle/input'))

assert CACHE_INPUT_DIR.exists(), f"❌ CACHE_INPUT_DIR not found: {CACHE_INPUT_DIR}"
assert MANIFEST_PATH.exists(),   f"❌ manifest.csv not found at {MANIFEST_PATH}"
assert NPY_CACHE_DIR.exists(),   f"❌ cache directory not found at {NPY_CACHE_DIR}"
assert STATS_PATH.exists(),      f"❌ normalization_stats.json not found"

print("✅ NB02 dataset found successfully.")

# Check number of cached files
npy_files = list(NPY_CACHE_DIR.glob("*.npy"))
print(f"NPY files found: {len(npy_files)}")

assert len(npy_files) > 0, "❌ No NPY files detected in cache directory."

print("✅ Cache directory verified.")

# Load normalization stats
with open(STATS_PATH) as f:

    norm_stats = json.load(f)

print("Std :", norm_stats["std"])

print("Normalization stats loaded:")
print("Mean:", norm_stats["mean"])

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────
import os, gc, time, random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
# AMP: use torch.amp.autocast / torch.amp.GradScaler (non-deprecated)

import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import (
    roc_auc_score, confusion_matrix,
    roc_curve, precision_recall_curve
)
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


def seed_everything(s: int):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)
print('Imports OK.')

In [ ]:
# ── 2. Dataset ────────────────────────────────────────────────────────────
import json as _json

SUBTYPES = ['any', 'epidural', 'intraparenchymal',
            'intraventricular', 'subarachnoid', 'subdural']

# ─── Load dataset-specific normalization stats (with ImageNet fallback) ──
_norm_path = os.path.join(CACHE_INPUT_DIR, 'normalization_stats.json')
if os.path.exists(_norm_path):
    with open(_norm_path) as f:
        _norm = _json.load(f)
    MEAN = _norm['mean']
    STD  = _norm['std']
    print(f'Loaded dataset-specific normalization: mean={MEAN}, std={STD}')
else:
    MEAN = [0.485, 0.456, 0.406]    # ImageNet fallback
    STD  = [0.229, 0.224, 0.225]
    print(f'WARNING: normalization_stats.json not found. Using ImageNet defaults.')
    print(f'  mean={MEAN}, std={STD}')


class ICHDataset(Dataset):
    """Loads cached NPY arrays (uint8 [0,255], H×W×3) and returns (image, label) pairs.

    For binary detection of *any* hemorrhage.  If multi-label targets
    are needed later, change `label_col` or return the full 6-dim vector.
    """

    def __init__(self, df: pd.DataFrame, npy_root: str, transform, label_col: str = 'any'):
        self.df        = df.reset_index(drop=True)
        self.npy_root  = npy_root
        self.transform = transform
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = os.path.join(self.npy_root, f'{row["image_id"]}.npy')
        try:
            img = np.load(path)                        # uint8 H×W×3 [0,255]
        except Exception:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        img  = self.transform(img)                     # → Tensor C×H×W normalised
        label = torch.tensor(float(row[self.label_col]), dtype=torch.float32)
        return img, label


# ─── Transforms ──────────────────────────────────────────────────────────
train_transform = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(),
    T.RandomRotation(degrees=10),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

val_transform = T.Compose([
    T.ToPILImage(),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

print('Dataset class + transforms defined.')

In [ ]:
# ── 3. Load manifest and build DataLoaders ────────────────────────────────
manifest = pd.read_csv(MANIFEST_PATH)
print(f'Manifest loaded: {len(manifest):,} rows')
print(f'Cache files: {len(os.listdir(NPY_CACHE_DIR)):,} .npy files found')

train_df = manifest[manifest['split'] == 'train'].reset_index(drop=True)
val_df   = manifest[manifest['split'] == 'val'].reset_index(drop=True)
print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}')

# ─── Weighted sampler to handle class imbalance ──────────────────────────
from torch.utils.data import WeightedRandomSampler

pos_count = int(train_df['any'].sum())
neg_count = len(train_df) - pos_count
pos_weight_val = neg_count / pos_count     # for BCEWithLogitsLoss
print(f'Pos: {pos_count:,}  Neg: {neg_count:,}  pos_weight={pos_weight_val:.2f}')

sample_weights = np.where(train_df['any'].values == 1,
                          neg_count / pos_count,
                          1.0)
sampler = WeightedRandomSampler(
    weights=sample_weights.tolist(),
    num_samples=len(train_df),
    replacement=True
)

train_ds = ICHDataset(train_df, NPY_CACHE_DIR, train_transform)
val_ds   = ICHDataset(val_df,   NPY_CACHE_DIR, val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Batches per epoch — train: {len(train_loader)}, val: {len(val_loader)}')

In [ ]:
# ── 4. Model definition ───────────────────────────────────────────────────
def build_model(arch: str, pretrained: bool = True) -> nn.Module:
    """
    Build a binary classifier on top of EfficientNet-B0 or ResNet-50.
    Output: single logit (no sigmoid — use BCEWithLogitsLoss).
    """
    if arch == 'efficientnet_b0':
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        m = models.efficientnet_b0(weights=weights)
        in_features = m.classifier[1].in_features
        m.classifier = nn.Sequential(
            nn.Dropout(p=DROPOUT),
            nn.Linear(in_features, 1)
        )

    elif arch == 'resnet50':
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        m = models.resnet50(weights=weights)
        in_features = m.fc.in_features
        m.fc = nn.Sequential(
            nn.Dropout(p=DROPOUT),
            nn.Linear(in_features, 1)
        )

    else:
        raise ValueError(f'Unknown architecture: {arch}')

    return m


# ─── Backbone freeze / unfreeze helper ───────────────────────────────────
def set_backbone_frozen(model, arch, freeze: bool):
    """Freeze or unfreeze all backbone layers (keep classifier head trainable)."""
    head_names = ('classifier',) if arch == 'efficientnet_b0' else ('fc',)
    for name, p in model.named_parameters():
        if not any(h in name for h in head_names):
            p.requires_grad = not freeze
    tag = 'FROZEN' if freeze else 'UNFROZEN'
    n_locked = sum(1 for p in model.parameters() if not p.requires_grad)
    print(f'  Backbone {tag} — {n_locked} parameter tensors locked')


model = build_model(ARCH, pretrained=True).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model    : {ARCH}')
print(f'Params   : {total_params:,}  (trainable: {train_params:,})')


In [ ]:
# ── 5. Loss, optimiser, scheduler ─────────────────────────────────────────
pos_weight_tensor = torch.tensor([pos_weight_val], device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

# ─── Discriminative LR: backbone trains slower than classifier head ──────
head_names = ('classifier',) if ARCH == 'efficientnet_b0' else ('fc',)
backbone_params = [p for n, p in model.named_parameters()
                   if not any(h in n for h in head_names)]
head_params     = [p for n, p in model.named_parameters()
                   if any(h in n for h in head_names)]
print(f'Param groups — backbone: {len(backbone_params)} tensors '
      f'(lr={BASE_LR * BACKBONE_LR_FACTOR:.1e}), '
      f'head: {len(head_params)} tensors (lr={BASE_LR:.1e})')

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': BASE_LR * BACKBONE_LR_FACTOR},
    {'params': head_params,     'lr': BASE_LR},
], weight_decay=WEIGHT_DECAY)

# CosineAnnealingLR over the TOTAL planned epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-6
)

scaler = torch.amp.GradScaler('cuda')   # mixed precision (non-deprecated)

print('Loss / optimizer / scheduler ready.')


In [ ]:
# ── 6. Load previous checkpoint (session chaining) ────────────────────────
START_EPOCH = 0
metrics_history = []
best_val_loss = float('inf')
patience_counter = 0
early_stopped = False

if PREV_CHECKPOINT_DIR is not None:
    ckpt_path = Path(PREV_CHECKPOINT_DIR) / 'checkpoint.pth'
    log_path  = Path(PREV_CHECKPOINT_DIR) / 'training_metrics.csv'

    if ckpt_path.exists():
        print(f'Loading checkpoint: {ckpt_path}')
        ckpt = torch.load(str(ckpt_path), map_location=DEVICE)

        model.load_state_dict(ckpt['model_state_dict'])
        START_EPOCH = ckpt['epoch'] + 1

        # Check optimizer compatibility (param group count must match)
        saved_groups = len(ckpt['optimizer_state_dict']['param_groups'])
        if saved_groups == len(optimizer.param_groups):
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
            scaler.load_state_dict(ckpt['scaler_state_dict'])
        else:
            print(f'⚠ Optimizer layout changed ({saved_groups} → '
                  f'{len(optimizer.param_groups)} param groups) — '
                  f'model weights loaded, optimizer/scheduler reset.')

        # Restore early-stopping state (backward-compatible)
        best_val_loss    = ckpt.get('best_val_loss', float('inf'))
        patience_counter = ckpt.get('patience_counter', 0)

        # Check if previous session already triggered early stopping
        if ckpt.get('early_stopped', False):
            print('⚠ Previous session already triggered early stopping.')
            print('  Set PATIENCE higher or adjust hyperparams to continue.')

        print(f'Resuming from epoch {START_EPOCH}  '
              f'(best_val_loss={best_val_loss:.5f}, '
              f'patience={patience_counter}/{PATIENCE})')
    else:
        print(f'WARNING: checkpoint not found at {ckpt_path}. Starting from scratch.')

    if log_path.exists():
        prev_log = pd.read_csv(log_path)
        metrics_history = prev_log.to_dict('records')
        print(f'Loaded {len(metrics_history)} previous epoch records')
        # Fallback: compute best_val_loss from history if not in checkpoint
        if best_val_loss == float('inf') and 'val_loss' in prev_log.columns:
            best_val_loss = float(prev_log['val_loss'].min())
            print(f'  (best_val_loss inferred from log: {best_val_loss:.5f})')
else:
    print('No previous checkpoint. Starting from scratch (epoch 0).')

# ─── Freeze backbone if still within the freeze period ────────────────────
if START_EPOCH < FREEZE_BACKBONE_EPOCHS:
    set_backbone_frozen(model, ARCH, freeze=True)
    print(f'🧊 Backbone frozen for epochs 0–{FREEZE_BACKBONE_EPOCHS - 1} (head-only training)')
else:
    print(f'Backbone trainable (freeze period was epochs 0–{FREEZE_BACKBONE_EPOCHS - 1})')

END_EPOCH = START_EPOCH + N_EPOCHS_THIS_SESSION
print(f'This session: epoch {START_EPOCH} → {END_EPOCH - 1}  (of 0-{TOTAL_EPOCHS-1} total)')


In [ ]:
# ── 7. Evaluation helpers ─────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    """Run inference on loader and return metrics dict."""
    model.eval()
    all_logits, all_labels = [], []

    for imgs, labels in tqdm(loader, desc='Val', leave=False):
        imgs   = imgs.to(DEVICE)
        with torch.amp.autocast('cuda'):
            logits = model(imgs).squeeze(1)   # (B,)
        all_logits.append(logits.cpu().float())
        all_labels.append(labels)

    all_logits = torch.cat(all_logits).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_probs  = torch.sigmoid(torch.tensor(all_logits)).numpy()

    # Loss
    val_loss = nn.BCEWithLogitsLoss()(
        torch.tensor(all_logits),
        torch.tensor(all_labels)
    ).item()

    # AUC
    auc = roc_auc_score(all_labels, all_probs)

    # Sensitivity & Specificity at Youden-optimal threshold
    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
    youden_idx = np.argmax(tpr - fpr)
    best_thresh = float(thresholds[youden_idx])
    preds = (all_probs >= best_thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(all_labels, preds).ravel()
    sensitivity  = tp / (tp + fn + 1e-9)
    specificity  = tn / (tn + fp + 1e-9)
    precision    = tp / (tp + fp + 1e-9)
    f1           = 2 * precision * sensitivity / (precision + sensitivity + 1e-9)

    return {
        'val_loss'    : round(val_loss, 5),
        'val_auc'     : round(float(auc), 5),
        'sensitivity' : round(float(sensitivity), 5),
        'specificity' : round(float(specificity), 5),
        'precision'   : round(float(precision), 5),
        'f1'          : round(float(f1), 5),
        'best_thresh' : round(float(best_thresh), 4),
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
        'all_probs'   : all_probs,
        'all_labels'  : all_labels,
        'fpr': fpr, 'tpr': tpr,
    }


print('Evaluation function defined.')

In [ ]:
# ── 8. Training loop ──────────────────────────────────────────────────────
best_auc = max((r['val_auc'] for r in metrics_history), default=0.0)

for epoch in range(START_EPOCH, END_EPOCH):
    # ── Unfreeze backbone at the scheduled epoch ──────────────────────
    if epoch == FREEZE_BACKBONE_EPOCHS:
        set_backbone_frozen(model, ARCH, freeze=False)
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'🔓 Backbone unfrozen at epoch {epoch} — {trainable:,} trainable params')

    epoch_start = time.time()
    model.train()

    train_loss  = 0.0
    n_batches   = 0
    nan_count   = 0   # track NaN/Inf losses

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{TOTAL_EPOCHS-1} [train]', leave=True)
    for imgs, labels in pbar:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            logits = model(imgs).squeeze(1)        # (B,)
            loss   = criterion(logits, labels)

        # ── NaN / Inf divergence guard ────────────────────────────────
        if not torch.isfinite(loss):
            nan_count += 1
            if nan_count >= 5:
                raise RuntimeError(
                    f'Training diverged: {nan_count} NaN/Inf losses in epoch {epoch}. '
                    f'Try reducing lr ({BASE_LR}) or batch size ({BATCH_SIZE}).'
                )
            print(f'  ⚠ NaN/Inf loss at batch {n_batches} — skipping update')
            optimizer.zero_grad(set_to_none=True)
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        n_batches  += 1
        pbar.set_postfix(loss=f'{train_loss/n_batches:.4f}')

    scheduler.step()

    if n_batches == 0:
        raise RuntimeError(f'No valid batches in epoch {epoch} — all losses were NaN/Inf!')

    train_loss /= n_batches
    metrics = evaluate(model, val_loader)

    elapsed = time.time() - epoch_start
    lrs = scheduler.get_last_lr()
    row = {
        'epoch'       : epoch,
        'train_loss'  : round(train_loss, 5),
        'lr_backbone' : lrs[0],
        'lr_head'     : lrs[1] if len(lrs) > 1 else lrs[0],
        'elapsed_s'   : round(elapsed, 1),
        'nan_batches' : nan_count,
        **{k: v for k, v in metrics.items()
           if k not in ('all_probs', 'all_labels', 'fpr', 'tpr')}
    }
    metrics_history.append(row)

    # ── Save metrics log ──────────────────────────────────────────────────
    pd.DataFrame(metrics_history).to_csv(METRICS_LOG_PATH, index=False)

    # ── Early stopping check ──────────────────────────────────────────────
    if metrics['val_loss'] < best_val_loss:
        best_val_loss = metrics['val_loss']
        patience_counter = 0
    else:
        patience_counter += 1

    # ── Save checkpoint (always, after every epoch) ───────────────────────
    torch.save({
        'epoch'                : epoch,
        'arch'                 : ARCH,
        'model_state_dict'     : model.state_dict(),
        'optimizer_state_dict' : optimizer.state_dict(),
        'scheduler_state_dict' : scheduler.state_dict(),
        'scaler_state_dict'    : scaler.state_dict(),
        'best_thresh'          : metrics['best_thresh'],
        'best_val_loss'        : best_val_loss,
        'patience_counter'     : patience_counter,
        'early_stopped'        : False,
    }, CHECKPOINT_PATH)

    # ── Save best model separately ────────────────────────────────────────
    if metrics['val_auc'] > best_auc:
        best_auc = metrics['val_auc']
        torch.save(model.state_dict(), '/kaggle/working/best_model.pth')
        print(f'  ★ New best AUC: {best_auc:.5f} — saved best_model.pth')

    frozen_tag = ' [head-only]' if epoch < FREEZE_BACKBONE_EPOCHS else ''
    print(f'  Epoch {epoch:02d}{frozen_tag} | '
          f'train_loss={train_loss:.4f} | '
          f'val_loss={metrics["val_loss"]:.4f} | '
          f'AUC={metrics["val_auc"]:.4f} | '
          f'Sens={metrics["sensitivity"]:.4f} | '
          f'Spec={metrics["specificity"]:.4f} | '
          f'patience={patience_counter}/{PATIENCE} | '
          f'NaN={nan_count} | '
          f'{elapsed:.0f}s')

    # ── Break if patience exhausted ───────────────────────────────────────
    if patience_counter >= PATIENCE:
        print(f'\n⛔ Early stopping at epoch {epoch} '
              f'(no val_loss improvement for {PATIENCE} epochs)')
        early_stopped = True
        # Update checkpoint to record early stopping
        ckpt_es = torch.load(CHECKPOINT_PATH, map_location='cpu')
        ckpt_es['early_stopped'] = True
        torch.save(ckpt_es, CHECKPOINT_PATH)
        break

print('\nSession training complete.')
if early_stopped:
    print(f'  ⛔ Stopped early at epoch {epoch} (best val_loss={best_val_loss:.5f})')
else:
    print(f'  Completed epochs {START_EPOCH}–{END_EPOCH - 1}')


In [ ]:
# ── 9. Plot learning curves ───────────────────────────────────────────────
log = pd.read_csv(METRICS_LOG_PATH)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(log['epoch'], log['train_loss'], label='Train loss')
axes[0].plot(log['epoch'], log['val_loss'],   label='Val loss')
axes[0].set(title='Loss', xlabel='Epoch', ylabel='BCE Loss')
axes[0].legend()

axes[1].plot(log['epoch'], log['val_auc'], color='tab:orange')
axes[1].set(title='Validation AUC-ROC', xlabel='Epoch', ylabel='AUC')
axes[1].set_ylim([0.5, 1.0])

axes[2].plot(log['epoch'], log['sensitivity'], label='Sensitivity')
axes[2].plot(log['epoch'], log['specificity'], label='Specificity')
axes[2].plot(log['epoch'], log['f1'],          label='F1')
axes[2].set(title='Sensitivity / Specificity / F1', xlabel='Epoch')
axes[2].legend()

plt.tight_layout()
plt.savefig('/kaggle/working/learning_curves.png', bbox_inches='tight')
plt.show()

print(log[['epoch', 'train_loss', 'val_loss', 'val_auc',
           'sensitivity', 'specificity', 'f1']].to_string(index=False))

In [ ]:
# ── 10. ROC curve for last validation run ────────────────────────────────
# Rerun evaluate on best model to get ROC arrays
best_model_path = '/kaggle/working/best_model.pth'
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
val_metrics = evaluate(model, val_loader)

plt.figure(figsize=(6, 6))
plt.plot(val_metrics['fpr'], val_metrics['tpr'],
         label=f'AUC = {val_metrics["val_auc"]:.4f}', color='tab:blue')
plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
plt.scatter([1 - val_metrics['specificity']],
            [val_metrics['sensitivity']],
            color='red', zorder=5,
            label=f'Youden pt (Thr={val_metrics["best_thresh"]:.3f})')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve (Best Model)')
plt.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/roc_curve.png', bbox_inches='tight')
plt.show()

print(f'Best model AUC   : {val_metrics["val_auc"]:.5f}')
print(f'Sensitivity      : {val_metrics["sensitivity"]:.5f}')
print(f'Specificity      : {val_metrics["specificity"]:.5f}')
print(f'F1               : {val_metrics["f1"]:.5f}')
print(f'Optimal threshold: {val_metrics["best_thresh"]:.4f}')

In [ ]:
# ── 11. Confusion matrix ──────────────────────────────────────────────────
import seaborn as sns

cm = np.array([[val_metrics['tn'], val_metrics['fp']],
               [val_metrics['fn'], val_metrics['tp']]])

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Neg', 'Pred Pos'],
            yticklabels=['True Neg', 'True Pos'])
plt.title(f'Confusion Matrix (threshold={val_metrics["best_thresh"]:.3f})')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', bbox_inches='tight')
plt.show()

print('\nOutputs saved to /kaggle/working/')
print('Files: checkpoint.pth, best_model.pth, training_metrics.csv,'
      ' learning_curves.png, roc_curve.png, confusion_matrix.png')
print()
print('NEXT STEP:')
print(' Save Version → Save & Run All → commit this output')
print(' In the next session: set PREV_CHECKPOINT_DIR to this notebook output path')

In [ ]:
# ── HEALTH CHECK — automated output validation ────────────────────────────
import json as _json_hc

errors = []

# Check checkpoint
if not os.path.exists(CHECKPOINT_PATH):
    errors.append('checkpoint.pth is MISSING')
else:
    ckpt_hc = torch.load(CHECKPOINT_PATH, map_location='cpu')
    if 'model_state_dict' not in ckpt_hc:
        errors.append('checkpoint.pth missing model_state_dict')
    if 'best_thresh' not in ckpt_hc:
        errors.append('checkpoint.pth missing best_thresh')
    ckpt_epoch = ckpt_hc.get('epoch', -1)
    expected_epoch = END_EPOCH - 1
    if ckpt_hc.get('early_stopped', False):
        print(f'  ℹ Training stopped early at epoch {ckpt_epoch}')
    elif ckpt_epoch != expected_epoch:
        errors.append(f'checkpoint epoch={ckpt_epoch} != expected {expected_epoch}')

# Check best model
if not os.path.exists('/kaggle/working/best_model.pth'):
    errors.append('best_model.pth is MISSING')

# Check metrics log
if not os.path.exists(METRICS_LOG_PATH):
    errors.append('training_metrics.csv is MISSING')
else:
    log_hc = pd.read_csv(METRICS_LOG_PATH)
    if not early_stopped and len(log_hc) < N_EPOCHS_THIS_SESSION:
        errors.append(f'metrics log has {len(log_hc)} rows, expected >= {N_EPOCHS_THIS_SESSION}')
    last_auc = log_hc['val_auc'].iloc[-1]
    if last_auc < 0.55:
        errors.append(f'Final AUC={last_auc:.4f} is suspiciously low — possible training issue')
    nan_total = log_hc.get('nan_batches', pd.Series([0])).sum()
    if nan_total > 10:
        errors.append(f'{nan_total} NaN batches detected across all epochs')

# Check plots
for plot in ['learning_curves.png', 'roc_curve.png', 'confusion_matrix.png']:
    if not os.path.exists(f'/kaggle/working/{plot}'):
        errors.append(f'Missing plot: {plot}')

actual_last = metrics_history[-1]['epoch'] if metrics_history else START_EPOCH
health = {
    'notebook'          : '03_train_session',
    'status'            : 'PASS' if not errors else 'FAIL',
    'errors'            : errors,
    'session_epochs'    : f'{START_EPOCH}-{actual_last}',
    'early_stopped'     : early_stopped,
    'best_auc'          : round(best_auc, 5),
    'best_val_loss'     : round(best_val_loss, 5) if best_val_loss < float('inf') else None,
    'final_train_loss'  : round(metrics_history[-1]['train_loss'], 5) if metrics_history else None,
}

with open('/kaggle/working/health_check_nb03.json', 'w') as f:
    _json_hc.dump(health, f, indent=2)

if errors:
    print('❌ HEALTH CHECK FAILED:')
    for e in errors:
        print(f'   • {e}')
else:
    print('✅ HEALTH CHECK PASSED')
    print(f'   Epochs trained : {START_EPOCH} → {actual_last}'
          + (' (early stopped)' if early_stopped else ''))
    print(f'   Best AUC       : {best_auc:.5f}')
    print(f'   Best val_loss  : {best_val_loss:.5f}')
    print(f'   Checkpoint     : saved')
    print(f'   Metrics log    : {len(log_hc)} rows')
